# 🌱 SoilNet – Soil Type Classification

---

## Problem Statement

Soil classification plays an important role in construction, agriculture, and geotechnical analysis.  
In many practical scenarios, preliminary soil inspection is still performed manually and depends heavily on field experience.

As someone coming from a civil engineering background, I found this problem interesting because soil identification is often one of the first steps in site investigation and material assessment. Traditional testing methods are reliable, but they can also be time-consuming and dependent on laboratory access.

This project explores whether a lightweight computer-vision system can assist in identifying soil categories from images using deep learning.

The goal is not to replace laboratory testing or geotechnical surveys, but to investigate how machine learning can support faster preliminary analysis in low-resource environments.

---

## Project Objective

Build a lightweight image-classification pipeline capable of distinguishing between 7 soil categories using transfer learning.

The system is intentionally designed for:

- CPU-only training and inference
- reproducible experimentation on consumer hardware
- lightweight deployment using FastAPI, Streamlit, Docker, and Hugging Face Spaces

This project also serves as a practical exploration of deep learning from the perspective of a civil engineering graduate transitioning into machine learning engineering.

---

## Engineering Constraints

One important constraint in this project was hardware accessibility.

The entire training and evaluation pipeline was developed on a laptop without a dedicated GPU. Because of this, model selection focused not only on accuracy, but also on:

- training stability on CPU
- smaller model size
- faster inference
- reproducibility on low-resource systems

This constraint influenced the decision to use transfer learning with EfficientNetV2B0 instead of training a large custom CNN from scratch.

---

## Dataset

Dataset source:  
Comprehensive Soil Classification Dataset from Kaggle.

The dataset contains soil images grouped into seven categories:

- Alluvial
- Black
- Laterite
- Red
- Yellow
- Arid
- Mountain

The dataset is relatively small and visually inconsistent in terms of:

- lighting conditions
- texture scale
- camera quality
- moisture variation

Because of this, lightweight augmentation techniques were introduced during training to improve generalization.

---

## Why EfficientNetV2B0?

A custom CNN was initially considered as a baseline approach.  
However, the dataset size (~1,000 images) is relatively small for training a deep convolutional architecture from scratch.

EfficientNetV2B0 was selected because it provides:

- strong feature extraction from ImageNet pretraining
- relatively low computational cost
- good accuracy-to-parameter efficiency
- practical CPU inference performance

The transfer-learning approach allowed the project to focus on deployment and evaluation rather than spending excessive compute resources on training.

---

## Notebook Roadmap

This notebook follows a complete deep-learning workflow:


1. Setup & Imports
2. Dataset Loading
3. Exploratory Data Analysis
4. Dataset Pipeline
5. Model Architecture
6. Initial Training
7. Fine-Tuning
8. Evaluation
9. Model Export
10. Inference Benchmarking
11. Conclusion

The notebook is intentionally written as a practical engineering document rather than a collection of disconnected code cells.

#  1. Setup & Imports

In [ ]:
import os
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split


warnings.filterwarnings("ignore")

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Plot settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("TensorFlow Version:", tf.__version__)

#2. Dataset Loading

# 📂 Dataset

This project uses the publicly available 7-class soil image dataset from Kaggle.

Dataset:
https://www.kaggle.com/datasets/ai4a-lab/comprehensive-soil-classification-datasets

The dataset contains images across multiple soil categories with variations in:
- texture
- lighting
- moisture
- camera angles
- environmental conditions

This variability makes the classification task more realistic and closer to field conditions.

---

## 2.1 Google Colab Setup

Since uploading large datasets directly to Colab can be slow and unstable, the dataset is loaded from Google Drive after extraction.

Expected folder structure:

data/\
└── soil_dataset/\
    ├── Alluvial_Soil/\
    ├── Arid_Soil/\
    ├── Black_Soil/\
    ├── Laterite_Soil/\
    ├── Mountain_Soil/\
    ├── Red_Soil/\
    └── Yellow_Soil/

In [ ]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

## 2.2  Dataset Acquisition
Instead of manually uploading thousands of images into Google Colab, the dataset is downloaded programmatically using the Kaggle API.

This approach makes the notebook:
- reproducible
- faster to set up
- easier to share
- closer to real-world ML workflows

The dataset used in this project is:

**Comprehensive Soil Classification Dataset**
from Kaggle.

This dataset contains multiple soil categories collected under varying environmental conditions, making it suitable for testing a lightweight computer-vision pipeline.

In [ ]:
# Install Kaggle API
!pip install -q kaggle

## Kaggle API Setup

To access Kaggle datasets securely, upload your `kaggle.json` API token.

You can generate this token from:
Kaggle → Account Settings → Create New API Token

In [ ]:
from google.colab import files
files.upload()

In [ ]:
# Configure Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download dataset
!kaggle datasets download -d ai4a-lab/comprehensive-soil-classification-datasets

In [ ]:
# Extract dataset

!unzip -q comprehensive-soil-classification-datasets.zip -d soil_data

DATA_DIR = "/content/soil_data/Orignal-Dataset"

## 2.3  Data Validation

Real-world datasets often contain:
- corrupted images
- incomplete files
- inconsistent formats

Before training the model, we validate the dataset to avoid runtime failures during TensorFlow data loading.

In [ ]:
from PIL import Image

bad_files = []

for root, _, files in os.walk(DATA_DIR):
    for file in files:
        path = os.path.join(root, file)

        try:
            img = Image.open(path)
            img.verify()

        except Exception:
            bad_files.append(path)

print(f"Corrupted files found: {len(bad_files)}")

## 2.4 Dataset Inspection

Before training a neural network, it is important to inspect the structure and quality of the dataset.

The expected directory structure is:

data/\
└── soil_dataset/\
    ├── Alluvial/\
    ├── Black/\
    ├── Laterite/\
    ├── Red/\
    ├── Yellow/\
    ├── Arid/\
    └── Mountain/

In [ ]:
DATA_DIR = "/content/soil_data/Orignal-Dataset"

# Verify class folders
subdirs = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

print("Classes found:", subdirs)
print("Number of classes:", len(subdirs))

## 2.5 Loading the dataset

Instead of manually loading every image into memory, this project uses
``` python  
tf.keras.utils.image_dataset_from_directory()

```
##This method is:

* memory efficient
* scalable
* cleaner for production workflows
* compatible with TensorFlow pipelines

In [ ]:
# ============================================================
# Dataset Loading
# ============================================================

DATA_DIR = "/content/soil_data/Orignal-Dataset"

image_paths = []
labels = []

# Scan dataset folders
for class_name in sorted(os.listdir(DATA_DIR)):

    class_dir = os.path.join(DATA_DIR, class_name)

    if os.path.isdir(class_dir):

        for image_name in os.listdir(class_dir):

            image_paths.append(
                os.path.join(class_dir, image_name)
            )

            labels.append(class_name)

# Create dataframe
df = pd.DataFrame({

    "filepath": image_paths,
    "label": labels

})

print("Total Images:", len(df))
df.head()

In [ ]:
# ============================================================
# Stratified Train / Validation Split
# ============================================================

train_df, val_df = train_test_split(

    df,

    test_size=0.2,

    stratify=df["label"],

    random_state=42

)

print("Training Samples:", len(train_df))
print("Validation Samples:", len(val_df))

print("\nValidation Distribution:\n")

print(
    val_df["label"].value_counts()
)

In [ ]:
# ============================================================
# TensorFlow Dataset Pipeline
# ============================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

class_names = sorted(df["label"].unique())

label_to_index = {

    name: idx
    for idx, name in enumerate(class_names)

}

train_df["label_idx"] = train_df["label"].map(label_to_index)
val_df["label_idx"] = val_df["label"].map(label_to_index)

In [ ]:
# ============================================================
# Image Loading Function
# ============================================================

def load_image(path, label):

    image = tf.io.read_file(path)

    # Decode image safely
    image = tf.image.decode_image(

        image,

        channels=3,

        expand_animations=False   # IMPORTANT FIX

    )

    # Set static shape
    image.set_shape([None, None, 3])

    # Resize
    image = tf.image.resize(

        image,

        IMG_SIZE

    )

    # Convert to float32
    image = tf.cast(

        image,

        tf.float32

    )

    return image, label

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices(

    (
        train_df["filepath"].values,
        train_df["label_idx"].values
    )

)

val_ds = tf.data.Dataset.from_tensor_slices(

    (
        val_df["filepath"].values,
        val_df["label_idx"].values
    )

)

train_ds = (
    train_ds
    .shuffle(1000)
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Dataset pipeline ready.")

# 3. Exploratory Data Analysis

Before training the model, it is important to understand:
- class balance
- image diversity
- texture variations
- visual overlap between categories

In many machine learning projects, poor understanding of the dataset leads to misleading evaluation results later.

This step helps validate whether the dataset is suitable for training.

##Class Distribution

In [ ]:
# ============================================================
# Class Distribution
# ============================================================

full_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_counts = {c: 0 for c in class_names}

for images, labels in full_ds:
    for label in labels.numpy():
        class_counts[class_names[label]] += 1

plt.figure(figsize=(10,5))

plt.bar(
    class_counts.keys(),
    class_counts.values()
)

plt.title("Number of Images per Soil Class")
plt.xlabel("Soil Type")
plt.ylabel("Image Count")
plt.xticks(rotation=45)

plt.show()

pd.DataFrame.from_dict(
    class_counts,
    orient='index',
    columns=['Count']
)

## 📊 Class Distribution Insights
The dataset shows noticeable class imbalance.

Observations:
- Arid Soil, Black Soil, and Laterite Soil contain significantly more images.
- Alluvial Soil and Yellow Soil contain comparatively fewer samples.
- This imbalance can affect model learning and bias predictions toward majority classes.

From a real-world perspective, this makes sense because:
- some soil types are easier to capture in large quantities
- some are geographically limited
- some have less visual diversity

Even though the dataset is relatively small for deep learning standards, transfer learning allows the model to generalize effectively without requiring millions of images.

## Visual Inspection of Samples

In [ ]:

def show_samples(dataset, class_names, n_per_class=3):

    collected = {i: [] for i in range(len(class_names))}

    for images, labels in dataset.unbatch():

        label = labels.numpy()

        if len(collected[label]) < n_per_class:
            collected[label].append(images.numpy())

        if all(len(collected[i]) >= n_per_class for i in range(len(class_names))):
            break

    plt.figure(figsize=(12, 12))

    for i, cls in enumerate(class_names):

        for j in range(min(n_per_class, len(collected[i]))):

            plt.subplot(
                len(class_names),
                n_per_class,
                i * n_per_class + j + 1
            )

            img = collected[i][j].astype("uint8")

            plt.imshow(img)
            plt.title(cls if j == 0 else "")
            plt.axis("off")

    plt.tight_layout()
    plt.show()

show_samples(full_ds, class_names)

## Visual Insights

The visual inspection reveals that the model is learning much more than simple color differences.

Examples:
- Black Soil contains dark, moisture-rich textures
- Arid Soil often contains cracked dry surfaces
- Laterite Soil shows coarse reddish granular patterns
- Mountain Soil includes rocky terrain and uneven textures

At the same time, several classes visually overlap:
- Red Soil and Laterite Soil share similar reddish tones
- Yellow Soil and Alluvial Soil sometimes appear visually similar under bright lighting

This makes the task challenging and realistic.

The model must learn:
- texture
- spatial patterns
- edge structures
- contrast variations

rather than memorizing colors alone.

# 4. Dataset Pipeline Engineering

During experimentation, an issue appeared where some classes were missing from validation metrics.

The root cause was not the model itself, but the dataset pipeline configuration.

When `shuffle=False` was used before splitting, TensorFlow preserved directory ordering. This caused certain classes to appear disproportionately in either training or validation subsets.

To fix this properly:
- shuffling was enabled during dataset splitting
- a separate non-shuffled evaluation dataset was created for deterministic evaluation

This reflects a real-world ML engineering lesson:
sometimes model issues are actually data pipeline issues.

In [ ]:
# ============================================================
# Dataset Optimization
# ============================================================

# Dataset is small enough to fit into RAM comfortably.
# Caching avoids repeated disk reads during training
# and improves CPU training speed significantly.

train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)

val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
# ============================================================
# Data Augmentation
# ============================================================

data_augmentation = keras.Sequential([

    # Horizontal flip improves robustness
    # against orientation changes
    layers.RandomFlip("horizontal"),

    # Small rotations simulate
    # real-world camera angles
    layers.RandomRotation(0.1),

    # Zoom augmentation helps the model
    # learn texture patterns at multiple scales
    layers.RandomZoom(0.1),

    # Contrast variation improves robustness
    # against lighting conditions in outdoor environments
    layers.RandomContrast(0.1)

])

## Why augmentation matters

In real field conditions:

- camera angles change
- lighting changes
- soil texture appears at different scales

Augmentation helps simulate these variations.

# 5. Model Architecture — EfficientNetV2B0

Initially, I considered building a CNN from scratch.

However, training a deep CNN from scratch requires:
- significantly larger datasets
- longer training times
- stronger hardware

Since this project was intentionally developed on a CPU-only laptop, transfer learning was a much more practical engineering decision.

EfficientNetV2B0 was selected because:
- lightweight architecture
- strong accuracy
- optimized parameter efficiency
- faster inference
- suitable for edge and mobile deployment

Instead of learning generic visual patterns from zero, the model reuses pretrained ImageNet features and adapts them for soil classification.

In [ ]:
# ============================================================
# Model Architecture
# ============================================================

def create_model(trainable_base=False):

    base = keras.applications.EfficientNetV2B0(
        include_top=False,
        weights='imagenet',
        input_shape=(224,224,3)
    )

    base.trainable = trainable_base

    inputs = keras.Input(shape=(224,224,3))

    x = data_augmentation(inputs)

    x = keras.applications.efficientnet_v2.preprocess_input(x)

    x = base(
        x,
        training=False if not trainable_base else None
    )

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        256,
        activation='relu'
    )(x)

    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(
        len(class_names),
        activation='softmax'
    )(x)

    model = keras.Model(inputs, outputs)

    return model

model = create_model(trainable_base=False)

model.summary()

In [ ]:
# ============================================================
# Model Compilation
# ============================================================

model.compile(

    optimizer=keras.optimizers.Adam(1e-3),

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

## Why Adam optimizer?

Adam is commonly used because it:

- converges quickly
- works well for transfer learning
- requires minimal tuning

This makes it suitable for lightweight experimentation workflows.

# 6. Initial Training

The first phase trains only the classifier head while keeping the EfficientNet backbone frozen.

This:
- reduces training time
- avoids destabilizing pretrained weights
- allows the classifier layers to adapt first

This approach is commonly used in transfer learning workflows.

In [ ]:
# ============================================================
# Handle Class Imbalance with Class Weights
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

# Extract labels from training dataset
train_labels = np.concatenate([
    y.numpy() for x, y in train_ds
])

# Compute balanced class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)

# Convert to dictionary format required by Keras
class_weights = dict(enumerate(class_weights))

print("Computed Class Weights:")
print(class_weights)

In [ ]:
# ============================================================
# Initial Training
# ============================================================

EPOCHS_HEAD = 15

history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=EPOCHS_HEAD,

    # Apply class balancing
    class_weight=class_weights

)

In [ ]:
# ============================================================
# Training Curves
# ============================================================

def plot_history(history):

    plt.figure(figsize=(12,4))

    # Accuracy
    plt.subplot(1,2,1)

    plt.plot(
        history.history['accuracy'],
        label='Train'
    )

    plt.plot(
        history.history['val_accuracy'],
        label='Validation'
    )

    plt.title('Accuracy')
    plt.legend()

    # Loss
    plt.subplot(1,2,2)

    plt.plot(
        history.history['loss'],
        label='Train'
    )

    plt.plot(
        history.history['val_loss'],
        label='Validation'
    )

    plt.title('Loss')
    plt.legend()

    plt.show()

plot_history(history)

## Initial Training Insights

The first training phase focused only on the custom classification head while keeping the EfficientNetV2B0 backbone frozen.

Training accuracy steadily improved from nearly 60% to around 94%, showing that the model successfully learned meaningful soil texture and color patterns from the dataset.

Validation accuracy stabilised around 84–86%, indicating good generalisation despite:
- limited dataset size,
- class imbalance,
- and visual similarity between certain soil categories.

The gap between training and validation accuracy suggests mild overfitting in later epochs, which is expected when training on relatively small image datasets.

Applying class weights during training helped the model pay more attention to minority soil categories such as:
- Alluvial Soil,
- and Yellow Soil,

resulting in more balanced performance across classes instead of optimising only for majority categories.

Overall, the initial transfer learning stage demonstrated that pretrained visual features from ImageNet transfer effectively to soil classification tasks, even when trained on a CPU-only environment.

# 7. Fine-Tuning

After the classifier head stabilized, the upper layers of EfficientNet were unfrozen for limited fine-tuning.

A very small learning rate is used because pretrained features are already highly optimized.

The goal is not dramatic relearning, but small feature adjustments specific to soil textures and patterns.

In [ ]:
# ============================================================
# Fine-Tuning
# ============================================================

base = model.get_layer("efficientnetv2-b0")

base.trainable = True

for layer in base.layers[:int(len(base.layers) * 0.8)]:
    layer.trainable = False

model.compile(

    optimizer=keras.optimizers.Adam(1e-5),

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

EPOCHS_FINE = 3

history_fine = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=EPOCHS_FINE

)

plot_history(history_fine)

## Fine-Tuning Insights

After the initial training phase, the top portion of the EfficientNetV2B0 backbone was unfrozen for limited fine-tuning.

A very small learning rate (`1e-5`) was used to slightly adjust pretrained ImageNet features without damaging previously learned representations.

During fine-tuning:
- training accuracy improved gradually,
- while validation accuracy remained relatively stable around 81%.

This behaviour suggests that the model had already learned most of the useful high-level visual features during the initial transfer learning phase.

The limited improvement during fine-tuning is expected because:
- the dataset is relatively small,
- several soil categories share visually overlapping textures,
- and aggressive fine-tuning on small datasets can quickly lead to overfitting.

Instead of aggressively optimising for marginal accuracy gains, the project prioritised:
- stable generalisation,
- lightweight training,
- and reproducible CPU-based experimentation.

This reflects a more practical engineering approach compared to excessive hyperparameter tuning on limited data.

# 8. Model Evaluation

Evaluation is performed on a separate non-shuffled validation dataset to ensure deterministic ordering.

The model is evaluated using:
- Precision
- Recall
- F1-score
- Confusion Matrix

These metrics provide a more complete understanding than accuracy alone.

In [ ]:
# ============================================================
# Classification Report
# ============================================================

y_true = np.concatenate([

    y.numpy()

    for x, y in val_ds

])

y_pred_probs = model.predict(val_ds)

y_pred = np.argmax(

    y_pred_probs,

    axis=1

)

report = classification_report(

    y_true,

    y_pred,

    target_names=class_names,

    zero_division=0

)

print(report)

## Evaluation Insights

The final model achieved approximately 81% validation accuracy with balanced performance across most soil categories.

Several classes such as:
- Black Soil,
- Arid Soil,
- Mountain Soil,
- and Yellow Soil

showed strong precision and recall, indicating that the model successfully learned distinctive visual patterns related to soil texture and color distribution.

The model performed particularly well on minority classes like Alluvial Soil after introducing class weighting during training. Recall improved noticeably compared to earlier experiments, showing that the balancing strategy helped reduce bias toward majority classes.

Some confusion still remained between visually similar soil categories such as:
- Laterite Soil,
- Red Soil,
- and Mountain Soil.

This is expected because these classes often share overlapping color tones, moisture conditions, and surface textures in real-world images.

Rather than aggressively tuning the model for benchmark accuracy, the project focused on:
- realistic evaluation,
- reproducible CPU-only training,
- lightweight deployment,
- and practical engineering tradeoffs.

The final results demonstrate that transfer learning can achieve strong and reliable performance even with:
- limited data,
- imbalanced classes,
- and consumer-grade hardware.

In [ ]:
# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(

    y_true,

    y_pred

)

plt.figure(figsize=(8,6))

sns.heatmap(

    cm,

    annot=True,

    fmt='d',

    cmap='Blues',

    xticklabels=class_names,

    yticklabels=class_names

)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("True")

plt.show()

## Confusion Matrix Analysis

The confusion matrix provides deeper insight into how the model behaves across individual soil categories.

Several classes achieved strong classification performance:
- Black Soil was identified consistently with very few misclassifications.
- Arid Soil and Mountain Soil also showed strong recall, indicating that the model learned robust texture-based features for these categories.
- Red Soil demonstrated high recall despite having fewer samples compared to majority classes.

The model showed the highest confusion in classes with overlapping visual characteristics:

- Laterite Soil was occasionally misclassified as:
  - Alluvial Soil,
  - Black Soil,
  - and Red Soil.

- Yellow Soil was sometimes confused with Arid Soil because both categories contain:
  - dry textures,
  - lighter color tones,
  - and similar surface patterns.

- Alluvial Soil remained challenging due to the very limited number of available training samples.

These results highlight a common real-world computer vision challenge:
fine-grained image classification becomes significantly harder when:
- datasets are small,
- classes are imbalanced,
- and visual boundaries between categories are not sharply defined.

Despite these challenges, the model maintained relatively balanced performance across most classes without relying on:
- excessive augmentation,
- synthetic oversampling,
- or computationally expensive architectures.

This demonstrates that transfer learning with EfficientNetV2B0 can provide strong practical performance even under constrained hardware and dataset conditions.

# 9. Model Export

After evaluation, the trained model and class labels are exported for deployment.

These artifacts will later be used by:
- FastAPI backend
- Streamlit frontend
- Docker container
- Hugging Face Spaces deployment

In [ ]:
# ============================================================
# Save Model Artifacts
# ============================================================

os.makedirs(
    "artifacts",
    exist_ok=True
)

model.save(
    "artifacts/model.keras"
)

with open(
    "artifacts/class_names.pkl",
    "wb"
) as f:

    pickle.dump(class_names, f)

print("Artifacts Saved Successfully")

# 10. Inference Benchmarking

In production systems, accuracy alone is not enough.

Inference latency and model size matter because:
- large models increase deployment cost
- slow inference hurts user experience
- lightweight models are easier to deploy on edge devices

This benchmark measures:
- single-image CPU inference latency
- model size on disk

In [ ]:
# ============================================================
# Inference Benchmarking
# ============================================================

sample = tf.random.normal((1, 224, 224, 3))

start = time.time()

_ = model.predict(sample)

latency = (time.time() - start) * 1000

print(f"Inference Latency: {latency:.2f} ms")

size_mb = os.path.getsize(
    "artifacts/model.keras"
) / (1024 * 1024)

print(f"Model Size: {size_mb:.2f} MB")

## Inference Benchmarking Insights

The exported model achieved an inference latency of approximately 10.3 seconds per image on a CPU-only environment, with a total model size of around 44 MB.

Although inference is slower than GPU-based systems, the project was intentionally designed and tested on consumer hardware without relying on cloud GPUs.

The current model size and performance are still practical for:
- Docker deployment,
- FastAPI inference APIs,
- and Hugging Face Spaces.

Future optimisation can include:
- TensorFlow Lite conversion,
- quantisation,
- or GPU deployment for faster real-time predictions.

# 11. Conclusion

This project explored the use of transfer learning for soil image classification using EfficientNetV2B0 on a CPU-only environment.

The final model achieved balanced performance across most soil categories despite:
- limited dataset size,
- class imbalance,
- and visually overlapping soil textures.

Instead of focusing only on benchmark accuracy, the project prioritised:
- practical deployment,
- lightweight experimentation,
- reproducible training,
- and realistic engineering constraints.

Several important machine learning concepts were applied throughout the workflow, including:
- exploratory data analysis,
- data augmentation,
- transfer learning,
- fine-tuning,
- class imbalance handling,
- performance evaluation,
- and inference benchmarking.

The project also demonstrates how domain knowledge from civil engineering can be combined with machine learning to solve practical real-world problems.

Most importantly, SoilNet was designed as an end-to-end system rather than just a notebook experiment. The next stage involves:
- exporting the trained model,
- building a FastAPI inference service,
- creating a Streamlit frontend,
- containerising the application with Docker,
- and deploying it on Hugging Face Spaces.

Overall, the project highlights a production-oriented approach to applied machine learning under realistic hardware and dataset constraints.

# 🚀 Next Steps

The next stage of the project focuses on productionization:

- FastAPI inference API
- Streamlit frontend
- Docker containerization
- Hugging Face deployment
- structured `src/` training pipeline
- recruiter-focused README

At this stage, the notebook experimentation phase is complete and the project transitions into deployment engineering.